# SAGE Phase 19 — component ablation

This notebook answers one question only: which parts of SAGE improve gradient fidelity or reduce exact solver work beyond compact perturb-and-project? It reuses the frozen Phase 18 checkpoints; it does **not** train more neural networks.

Before **Save Version → Run All**, attach `SAGE_Phase18_fixed_complete` as a Kaggle Input, enable Internet, select **No accelerator**, and keep the default 4 CPU threads. The run has per-center atomic checkpoints, no wall-time stopping rule, and no arbitrary performance pass/fail gate. Expected runtime from the local smoke extrapolation is roughly 1–3 hours, but the notebook reports only an informational ETA.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, tarfile, urllib.request, zipfile

REPO_URL = 'https://github.com/ghugn/SAGE_ICML2027.git'
REPO_BRANCH = 'main'
JULIA_VERSION = '1.13.3'
WORK = Path('/kaggle/working')
INPUT = Path('/kaggle/input')
REPO = WORK / 'SAGE_ICML2027'
DEPOT = WORK / 'julia_depot'
OUTPUT = WORK / 'SAGE_Phase19_component_ablation'

assert WORK.exists(), 'This notebook is intended for Kaggle.'
if (REPO / '.git').exists():
    subprocess.run(['git', 'pull', '--ff-only', 'origin', REPO_BRANCH], cwd=REPO, check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO)], check=True)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
assert (REPO / 'scripts' / 'benchmark_phase19_component_ablation.jl').exists(), 'Phase 19 code is not present in the cloned commit.'

selections = sorted(INPUT.rglob('phase18_selection.jls'))
if not selections:
    archives = sorted(INPUT.rglob('SAGE_Phase18_fixed_complete*.zip'))
    assert archives, 'Attach SAGE_Phase18_fixed_complete as an extracted folder or ZIP Kaggle Input.'
    restore = WORK / 'phase18_complete_restore'
    if restore.exists():
        shutil.rmtree(restore)
    restore.mkdir(parents=True)
    with zipfile.ZipFile(archives[-1], 'r') as bundle:
        bundle.extractall(restore)
    selections = sorted(restore.rglob('phase18_selection.jls'))
assert len(selections) == 1, f'Expected one Phase 18 result tree, found {len(selections)}.'
PHASE18_RESULTS = selections[0].parent
required = list((PHASE18_RESULTS / 'model_checkpoints').glob('seed_1[1-5]_*_fixed.jls'))
assert len(required) >= 30, f'Phase 18 bundle is incomplete: found {len(required)} fixed checkpoints.'
print('Code commit:', commit)
print('Phase 18 results:', PHASE18_RESULTS)

JULIA = WORK / f'julia-{JULIA_VERSION}' / 'bin' / 'julia'
if not JULIA.exists():
    archive = WORK / f'julia-{JULIA_VERSION}-linux-x86_64.tar.gz'
    url = f'https://julialang-s3.julialang.org/bin/linux/x64/1.13/julia-{JULIA_VERSION}-linux-x86_64.tar.gz'
    urllib.request.urlretrieve(url, archive)
    with tarfile.open(archive, 'r:gz') as bundle:
        bundle.extractall(WORK)
assert JULIA.exists(), JULIA
DEPOT.mkdir(parents=True, exist_ok=True)
env = os.environ.copy()
env.update({
    'JULIA_DEPOT_PATH': str(DEPOT),
    'JULIA_NUM_THREADS': '4',
    'OPENBLAS_NUM_THREADS': '1',
    'OMP_NUM_THREADS': '1',
    'SAGE_PHASE18_BUNDLE': str(PHASE18_RESULTS),
    'SAGE_PHASE19_OUTPUT': str(OUTPUT),
    'SAGE_PHASE19_PROFILE': 'confirm',
})
subprocess.run([str(JULIA), '--threads=4', '--project=.', '-e', 'using Pkg; Pkg.instantiate(); Pkg.precompile()'], cwd=REPO, env=env, check=True)
print('PHASE19_SETUP=PASS')

## Confirmatory diagnostic

Ten fixed centers are evaluated: one compact-black-box checkpoint center and one SAGE checkpoint center for each held-out seed 11–15. Six variants are compared one factor at a time with frozen `rho=1`, `N=4`, common random numbers for isotropic variants, 16 repetitions, and a 256-sample numerical reference.

In [ ]:
log_path = WORK / 'phase19_component_ablation.log'
command = [str(JULIA), '--threads=4', '--project=.', 'scripts/benchmark_phase19_component_ablation.jl']
with log_path.open('a', encoding='utf-8', buffering=1) as log:
    process = subprocess.Popen(command, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
        log.write(line)
    code = process.wait()
if code != 0:
    raise RuntimeError(f'Phase 19 exited with code {code}. Completed-center checkpoints remain in {OUTPUT}; rerun unchanged in the same session to resume.')
text = log_path.read_text(encoding='utf-8', errors='replace')
assert 'PHASE19_ABLATION=COMPLETE' in text, 'Completion token is missing.'
print('PHASE19_CONFIRMATORY=COMPLETE')

## Compact result and export

The table below is diagnostic, not an automatically selected winner. Download the ZIP from the saved Kaggle Version **Output** tab and add it to the local project for the final mentor report.

In [ ]:
import pandas as pd
from IPython.display import display

summary = json.loads((OUTPUT / 'phase19_summary.json').read_text(encoding='utf-8'))
assert summary['status'] == 'COMPLETE' and summary['safety_gate'] is True
table = pd.read_csv(OUTPUT / 'phase19_aggregate.csv')
display(table[['variant', 'mean_relative_bias', 'mean_trace_variance', 'mean_rmse', 'mean_cosine', 'mean_global_searches', 'mean_unique_branch_solves', 'time_ratio_vs_compact']])
archive = Path(shutil.make_archive(str(WORK / 'SAGE_Phase19_component_ablation_complete'), 'zip', root_dir=OUTPUT.parent, base_dir=OUTPUT.name))
print('Archive:', archive, f'({archive.stat().st_size / 1024**2:.2f} MB)')
print('Download from the saved Version Output tab.')
print('PHASE19_EXPORT=COMPLETE')